## Rules

| Case                                | Action                   |
| ----------------------------------- | ------------------------ |
| Same entities, exact same predicate | Exclude (true duplicate) |
| Same entities, paraphrase predicate | Accept + normalize       |
| Same entities, broader predicate    | Accept                   |
| Same entities, different relation   | Accept                   |
| Same entities, conflicting relation | Accept + flag            |


## My current entity resolution pipeline

A. Blocking (Generate Candidates)
- FAISS based blocking

B. Entity Resolution (Check similarity)
- String similarity
- Embedding similarity 
- Neighborhood similarity

C. Merging (Merge similar entities)
- Merge entities threshold is met
 

## My pipeline currently uses:
- FAISS blocking
- String similarity
- Neighbor overlap
- Entity typing (added)
- Contextual mention embeddings (added)
- Predicate semantics

it does not have:
- Sense disambiguation

## Upgrade road map

- Phase 1
    - Entity typing (even rule-based)
    - Predicate embedding clustering
    - Logging only (no splitting yet)

- Phase 2 
    - Contextual mention embeddings
    - Predicate-conditioned similarity

- Phase 3 
    - Mention-level entities
    - Sense-aware merging

##  To do

- Check why neighbor similarity is not working (its working)
- Convert this into a class-based pipeline
- Add FAISS-based blocking (done)
- Add entity embedding updates after merge

## Pipeline evolution

- String similarity
- Embeddings
- FAISS blocking
- Neighborhood similarity
- Entity typing
- Contextual mentions
- Predicate semantics


| Ambiguity              | Solution      | Complexity |
| ---------------------- | ------------------ | ---------------------- |
| Typos / abbreviations  | String sim         | Cheap                  |
| Synonyms / paraphrases | Embeddings         | Costly but general     |
| Alias vs same name     | FAISS              | Indexing + state       |
| Polysemy               | Typing             | Extra model            |
| Role-dependent meaning | Context embeddings | More inference         |
| Semantic drift         | Predicate modeling | Graph semantics        |


## Alternatives to similarity scoring

1. Learned merge classifier - Train a model so that it can learn when to merge instead of thresholding 
2. Cluster-level validation - Instead of pairwise similarity, evaluate merge of entity to cluster by comparing against cluster centroid embedding, context distribution, type distribution and relational signature
3. Relational Signature Modeling - For each entity mention, build a relational signature vector.

This vector represents
- the predicates an entity participates in.
- The direction of the participation (subject/object).
- The frequency or importance of each predicate.
 
4. Negative Evidence Modeling - 



In [101]:
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer, util

from generate_graph import get_propositions, generateEdges, createGraph, get_propositions_nosplit
from refine_graph_v4 import find_merge_candidates
from query_graph import QueryGraph
from tqdm import tqdm
from fuzzywuzzy import fuzz
from knowledge_graph_maker import Edge, Node

import pandas as pd
tqdm.pandas()

In [2]:
# model = SentenceTransformer("all-MiniLM-L6-v2")
model = SentenceTransformer("all-mpnet-base-v2")
# model = SentenceTransformer("E5-Base")

In [ ]:
# list_of_edges = [

#     # =========================
#     # 1. Homonym & Lexical Collision
#     # =========================
#     Edge(
#         node_1=Node(label="Organization", name="Apple", properties={}),
#         node_2=Node(label="Product", name="iPhone 15", properties={}),
#         relationship="released",
#         metadata={"subject_gold": "APPLE_TECH", "object_gold": "IPHONE_15"},
#         order=0
#     ),
#     Edge(
#         node_1=Node(label="Food", name="Apple", properties={}),
#         node_2=Node(label="Time", name="September", properties={}),
#         relationship="harvested in",
#         metadata={"subject_gold": "APPLE_FRUIT", "object_gold": "MONTH_SEPTEMBER"},
#         order=1
#     ),
#     Edge(
#         node_1=Node(label="Food", name="Apple", properties={}),
#         node_2=Node(label="FoodCategory", name="Fruit", properties={}),
#         relationship="is a type of",
#         metadata={"subject_gold": "APPLE_FRUIT", "object_gold": "FRUIT"},
#         order=2
#     ),
#     Edge(
#         node_1=Node(label="Organization", name="Apple Inc.", properties={}),
#         node_2=Node(label="Product", name="Macbook", properties={}),
#         relationship="sells",
#         metadata={"subject_gold": "APPLE_TECH", "object_gold": "MACBOOK"},
#         order=3
#     ),

#     # =========================
#     # 2. Abbreviation Ambiguity
#     # =========================
#     Edge(
#         node_1=Node(label="GeopoliticalEntity", name="US", properties={}),
#         node_2=Node(label="GeopoliticalEntity", name="Iraq", properties={}),
#         relationship="invaded",
#         metadata={"subject_gold": "UNITED_STATES", "object_gold": "IRAQ"},
#         order=4
#     ),
#     Edge(
#         node_1=Node(label="Person", name="US", properties={}),
#         node_2=Node(label="SportsTournament", name="US Open", properties={}),
#         relationship="won",
#         metadata={"subject_gold": "US_TENNIS_PLAYER", "object_gold": "US_OPEN_TOURNAMENT"},
#         order=5
#     ),
#     Edge(
#         node_1=Node(label="GeopoliticalEntity", name="United States", properties={}),
#         node_2=Node(label="City", name="Washington DC", properties={}),
#         relationship="has capital",
#         metadata={"subject_gold": "UNITED_STATES", "object_gold": "WASHINGTON_DC"},
#         order=6
#     ),

#     # =========================
#     # 3. Semantic Embedding Trap
#     # =========================
#     Edge(
#         node_1=Node(label="Organization", name="Amazon", properties={}),
#         node_2=Node(label="Product", name="Alexa", properties={}),
#         relationship="develops",
#         metadata={"subject_gold": "AMAZON_COMPANY", "object_gold": "ALEXA"},
#         order=7
#     ),
#     Edge(
#         node_1=Node(label="Organization", name="Amazon", properties={}),
#         node_2=Node(label="City", name="Seattle", properties={}),
#         relationship="headquartered in",
#         metadata={"subject_gold": "AMAZON_COMPANY", "object_gold": "SEATTLE"},
#         order=8
#     ),
#     Edge(
#         node_1=Node(label="River", name="Amazon River", properties={}),
#         node_2=Node(label="Country", name="Brazil", properties={}),
#         relationship="flows through",
#         metadata={"subject_gold": "AMAZON_RIVER", "object_gold": "BRAZIL"},
#         order=9
#     ),

#     # =========================
#     # 4. Role vs Entity Confusion
#     # =========================
#     Edge(
#         node_1=Node(label="Role", name="President", properties={}),
#         node_2=Node(label="GeopoliticalEntity", name="United States", properties={}),
#         relationship="leads",
#         metadata={"subject_gold": "US_PRESIDENT_ROLE", "object_gold": "UNITED_STATES"},
#         order=10
#     ),
#     Edge(
#         node_1=Node(label="Person", name="Joe Biden", properties={}),
#         node_2=Node(label="Role", name="President", properties={}),
#         relationship="is",
#         metadata={"subject_gold": "JOE_BIDEN", "object_gold": "US_PRESIDENT_ROLE"},
#         order=11
#     ),
#     Edge(
#         node_1=Node(label="Person", name="Joe Biden", properties={}),
#         node_2=Node(label="City", name="Scranton", properties={}),
#         relationship="born in",
#         metadata={"subject_gold": "JOE_BIDEN", "object_gold": "SCRANTON_PA"},
#         order=12
#     ),

#     # =========================
#     # 5. Predicate Paraphrase
#     # =========================
#     Edge(
#         node_1=Node(label="Person", name="Elon Musk", properties={}),
#         node_2=Node(label="Organization", name="SpaceX", properties={}),
#         relationship="leads",
#         metadata={"subject_gold": "ELON_MUSK", "object_gold": "SPACEX"},
#         order=13
#     ),
#     Edge(
#         node_1=Node(label="Person", name="Elon Musk", properties={}),
#         node_2=Node(label="Organization", name="SpaceX", properties={}),
#         relationship="is CEO of",
#         metadata={"subject_gold": "ELON_MUSK", "object_gold": "SPACEX"},
#         order=14
#     ),
#     Edge(
#         node_1=Node(label="Person", name="Elon Musk", properties={}),
#         node_2=Node(label="Organization", name="SpaceX", properties={}),
#         relationship="runs",
#         metadata={"subject_gold": "ELON_MUSK", "object_gold": "SPACEX"},
#         order=15
#     ),

#     # =========================
#     # 6. Predicate Polarity Conflict
#     # =========================
#     Edge(
#         node_1=Node(label="Organization", name="Company X", properties={}),
#         node_2=Node(label="Organization", name="Company Y", properties={}),
#         relationship="acquired",
#         metadata={"subject_gold": "COMPANY_X", "object_gold": "COMPANY_Y"},
#         order=16
#     ),
#     Edge(
#         node_1=Node(label="Organization", name="Company Y", properties={}),
#         node_2=Node(label="Organization", name="Company X", properties={}),
#         relationship="acquired",
#         metadata={"subject_gold": "COMPANY_Y", "object_gold": "COMPANY_X"},
#         order=17
#     ),

#     # =========================
#     # 7. Structural / Neighborhood Deception
#     # =========================
#     Edge(
#         node_1=Node(label="City", name="Paris", properties={}),
#         node_2=Node(label="Country", name="France", properties={}),
#         relationship="located in",
#         metadata={"subject_gold": "PARIS_FRANCE", "object_gold": "FRANCE"},
#         order=18
#     ),
#     Edge(
#         node_1=Node(label="City", name="Paris", properties={}),
#         node_2=Node(label="River", name="Seine", properties={}),
#         relationship="has river",
#         metadata={"subject_gold": "PARIS_FRANCE", "object_gold": "SEINE_RIVER"},
#         order=19
#     ),
#     Edge(
#         node_1=Node(label="City", name="Paris", properties={}),
#         node_2=Node(label="State", name="Texas", properties={}),
#         relationship="located in",
#         metadata={"subject_gold": "PARIS_TEXAS", "object_gold": "TEXAS"},
#         order=20
#     ),
#     Edge(
#         node_1=Node(label="City", name="Paris", properties={}),
#         node_2=Node(label="NumericValue", name="25000", properties={}),
#         relationship="has population",
#         metadata={"subject_gold": "PARIS_TEXAS", "object_gold": "POPULATION_25000"},
#         order=21
#     ),

#     # =========================
#     # 8. Copycat Organization
#     # =========================
#     Edge(
#         node_1=Node(label="Organization", name="OpenAI", properties={}),
#         node_2=Node(label="Product", name="ChatGPT", properties={}),
#         relationship="develops",
#         metadata={"subject_gold": "OPENAI", "object_gold": "CHATGPT"},
#         order=22
#     ),
#     Edge(
#         node_1=Node(label="Organization", name="OpenAI Research", properties={}),
#         node_2=Node(label="Product", name="ChatGPT", properties={}),
#         relationship="develops",
#         metadata={"subject_gold": "OPENAI_RESEARCH", "object_gold": "CHATGPT"},
#         order=23
#     ),
#     Edge(
#         node_1=Node(label="Organization", name="OpenAI Research", properties={}),
#         node_2=Node(label="Document", name="Paper A", properties={}),
#         relationship="published",
#         metadata={"subject_gold": "OPENAI_RESEARCH", "object_gold": "PAPER_A"},
#         order=24
#     ),
#     Edge(
#         node_1=Node(label="Organization", name="OpenAI", properties={}),
#         node_2=Node(label="Document", name="Paper B", properties={}),
#         relationship="published",
#         metadata={"subject_gold": "OPENAI", "object_gold": "PAPER_B"},
#         order=25
#     ),

#     # =========================
#     # 9. Temporal Drift
#     # =========================
#     Edge(
#         node_1=Node(label="Organization", name="Google", properties={}),
#         node_2=Node(label="Person", name="Larry Page", properties={}),
#         relationship="CEO",
#         metadata={"subject_gold": "GOOGLE", "object_gold": "LARRY_PAGE"},
#         order=26
#     ),
#     Edge(
#         node_1=Node(label="Organization", name="Google", properties={}),
#         node_2=Node(label="Person", name="Sundar Pichai", properties={}),
#         relationship="CEO",
#         metadata={"subject_gold": "GOOGLE", "object_gold": "SUNDAR_PICHAI"},
#         order=27
#     ),

#     # =========================
#     # 10. Compound Adversarial Case
#     # =========================
#     Edge(
#         node_1=Node(label="Organization", name="Meta", properties={}),
#         node_2=Node(label="Organization", name="Facebook", properties={}),
#         relationship="formerly known as",
#         metadata={"subject_gold": "META", "object_gold": "FACEBOOK"},
#         order=28
#     ),
#     Edge(
#         node_1=Node(label="Organization", name="Facebook", properties={}),
#         node_2=Node(label="Organization", name="Instagram", properties={}),
#         relationship="owns",
#         metadata={"subject_gold": "FACEBOOK", "object_gold": "INSTAGRAM"},
#         order=29
#     ),
#     Edge(
#         node_1=Node(label="Organization", name="Meta AI", properties={}),
#         node_2=Node(label="Model", name="LLaMA", properties={}),
#         relationship="develops",
#         metadata={"subject_gold": "META_AI", "object_gold": "LLAMA_MODEL"},
#         order=30
#     ),
#     Edge(
#         node_1=Node(label="Organization", name="Facebook AI Research", properties={}),
#         node_2=Node(label="Model", name="LLaMA", properties={}),
#         relationship="develops",
#         metadata={"subject_gold": "FAIR", "object_gold": "LLAMA_MODEL"},
#         order=31
#     )
# ]

In [84]:
from knowledge_graph_maker import Edge, Node

list_of_edges = [

    # =========================
    # 1. Homonym & Lexical Collision
    # =========================

    Edge(
        node_1=Node(
            label="Organization",
            name="Apple",
            properties={"entity_type": "technology_company"}
        ),
        node_2=Node(
            label="Product",
            name="iPhone 15",
            properties={}
        ),
        relationship="released",
        metadata={
            "summary": "Apple released the iPhone 15.",
            "subject_gold": "APPLE_TECH",
            "object_gold": "IPHONE_15",
            "generated_at": "2025-10-30 17:34:07.282620"
        },
        order=0
    ),

    Edge(
        node_1=Node(
            label="Food",
            name="Apple",
            properties={"entity_type": "fruit"}
        ),
        node_2=Node(
            label="Time",
            name="September",
            properties={}
        ),
        relationship="harvested in",
        metadata={
            "summary": "Apple is harvested in September.",
            "subject_gold": "APPLE_FRUIT",
            "object_gold": "MONTH_SEPTEMBER",
            "generated_at": "2025-10-30 17:34:07.282620"
        },
        order=1
    ),

    Edge(
        node_1=Node(
            label="Food",
            name="Apple",
            properties={"entity_type": "fruit"}
        ),
        node_2=Node(
            label="FoodCategory",
            name="Fruit",
            properties={}
        ),
        relationship="is a type of",
        metadata={
            "summary": "Apple is a type of fruit.",
            "subject_gold": "APPLE_FRUIT",
            "object_gold": "FRUIT",
            "generated_at": "2025-10-30 17:34:07.282620"
        },
        order=2
    ),

    Edge(
        node_1=Node(
            label="Organization",
            name="Apple Inc.",
            properties={"entity_type": "technology_company"}
        ),
        node_2=Node(
            label="Product",
            name="MacBook",
            properties={}
        ),
        relationship="sells",
        metadata={
            "summary": "Apple Inc. sells the MacBook.",
            "subject_gold": "APPLE_TECH",
            "object_gold": "MACBOOK",
            "generated_at": "2025-10-30 17:34:07.282620"
        },
        order=3
    )
]

In [85]:
toy_data = []
unique_entities = set()
for edge in list_of_edges:
    unique_entities.add(edge.node_1.name)
    unique_entities.add(edge.node_2.name)

    toy_data.append({
        "subject": edge.node_1.name,
        "subject_gold": edge.metadata.get("subject_gold"),
        "predicate": edge.relationship,
        "object": edge.node_2.name,     
        "object_gold": edge.metadata.get("object_gold"),
        # "context": edge.metadata.get("summary"),
    })

unique_entities

{'Apple', 'Apple Inc.', 'Fruit', 'MacBook', 'September', 'iPhone 15'}

In [86]:
toy_data

[{'subject': 'Apple',
  'subject_gold': 'APPLE_TECH',
  'predicate': 'released',
  'object': 'iPhone 15',
  'object_gold': 'IPHONE_15'},
 {'subject': 'Apple',
  'subject_gold': 'APPLE_FRUIT',
  'predicate': 'harvested in',
  'object': 'September',
  'object_gold': 'MONTH_SEPTEMBER'},
 {'subject': 'Apple',
  'subject_gold': 'APPLE_FRUIT',
  'predicate': 'is a type of',
  'object': 'Fruit',
  'object_gold': 'FRUIT'},
 {'subject': 'Apple Inc.',
  'subject_gold': 'APPLE_TECH',
  'predicate': 'sells',
  'object': 'MacBook',
  'object_gold': 'MACBOOK'}]

In [87]:
# from knowledge_graph_maker import Edge, Node

# list_of_edges2 = [Edge(node_1=Node(label='Person', name='John Doe', properties={'occupation': 'software engineer', 'rank': 'employee', 'birth_place': ''}), node_2=Node(label='Organization', name='Microsoft Corporation', properties={}), relationship='works at', metadata={'summary': 'John Doe works at Microsoft Corporation as a software engineer.', 'generated_at': '2025-10-30 17:34:07.282620'}, order=0),
#  Edge(node_1=Node(label='Person', name='J. Doe', properties={'occupation': 'employee'}), node_2=Node(label='Organization', name='Microsoft Corporation', properties={}), relationship='works at', metadata={'summary': 'J. Doe is an employee of Microsoft.', 'generated_at': '2025-10-30 17:34:07.282620'}, order=1),
 
#  Edge(node_1=Node(label='Person', name='Alice Smith', properties={'occupation': 'Employee', 'rank': 'N/A', 'birth_place': 'N/A'}), node_2=Node(label='Organization', name='Google LLC', properties={}), relationship='employee of', metadata={'summary': 'Alice Smith is employed by Google LLC.', 'generated_at': '2025-10-30 17:34:07.282620'}, order=2),
 
#  Edge(node_1=Node(label='Person', name='A. Smith', properties={'occupation': 'Employee'}), node_2=Node(label='Organization', name='Google', properties={}), relationship='works at', metadata={'summary': 'A. Smith works at Google.', 'generated_at': '2025-10-30 17:34:07.282620'}, order=3),
 
#  Edge(node_1=Node(label='Organization', name='Amazon', properties={}), node_2=Node(label='Object', name='Alexa', properties={}), relationship='develops', metadata={'summary': 'Amazon develops the voice assistant Alexa.', 'generated_at': '2025-10-30 17:34:07.282620'}, order=4),
 
#  Edge(node_1=Node(label='Organization', name='Amazon Inc.', properties={}), node_2=Node(label='Object', name='Alexa voice assistant', properties={}), relationship='produces', metadata={'summary': 'Amazon Inc. produces the Alexa voice assistant.', 'generated_at': '2025-10-30 17:34:07.282620'}, order=5),
 
#  Edge(node_1=Node(label='Person', name='Elon Musk', properties={'occupation': 'CEO', 'birth_place': 'Pretoria, South Africa'}), node_2=Node(label='Organization', name='SpaceX', properties={'type': 'Aerospace manufacturer', 'founded': '2002'}), relationship='leads', metadata={'summary': 'Elon Musk leads SpaceX.', 'generated_at': '2025-10-30 17:34:07.282620'}, order=6),
 
#  Edge(node_1=Node(label='Person', name='Elon Musk', properties={'occupation': 'CEO', 'birth_place': '', 'rank': ''}), node_2=Node(label='Organization', name='Space Exploration Technologies', properties={}), relationship='CEO of', metadata={'summary': 'Elon Musk is the CEO of Space Exploration Technologies.', 'generated_at': '2025-10-30 17:34:07.282620'}, order=7),
 
#  Edge(node_1=Node(label='Organization', name='Apple', properties={}), node_2=Node(label='Person', name='Steve Jobs', properties={'occupation': 'Entrepreneur', 'rank': 'Founder'}), relationship='founded by', metadata={'summary': 'Apple was founded by Steve Jobs.', 'generated_at': '2025-10-30 17:34:07.282620'}, order=8),
 
#  Edge(node_1=Node(label='Organization', name='Apple Inc.', properties={}), node_2=Node(label='Person', name='S. Jobs', properties={}), relationship='co-founded by', metadata={'summary': 'Apple Inc. was co-founded by S. Jobs.', 'generated_at': '2025-10-30 17:34:07.282620'}, order=9)]


# toy_data = []
# for edge in list_of_edges2:
# # unique_entities.add(edge.node_1.name)
# # unique_entities.add(edge.node_2.name)

#     toy_data.append({
#         "subject": edge.node_1.name,
#         "predicate": edge.relationship,
#         "object": edge.node_2.name     
#         # "context": edge.metadata.get("summary"),
#     })

## V6

Components

- FAISS embedding
- Semantic blocking
- String similarity
- Contextual Embedding similarity (using pre-trained model)
- Neighbor overlap
- Entity typing

In [88]:
# stress_test_dataset_with_labels = [

#     # =========================
#     # 1. Homonym & Lexical Collision
#     # =========================
#     {
#         "subject": "Apple",
#         "subject_gold": "APPLE_TECH",
#         "predicate": "released",
#         "object": "iPhone 15",
#         "object_gold": "IPHONE_15"
#     },
#     {
#         "subject": "Apple",
#         "subject_gold": "APPLE_FRUIT",
#         "predicate": "harvested in",
#         "object": "September",
#         "object_gold": "MONTH_SEPTEMBER"
#     },
#     {
#         "subject": "Apple",
#         "subject_gold": "APPLE_FRUIT",
#         "predicate": "is a type of",
#         "object": "Fruit",
#         "object_gold": "FRUIT"
#     },
#     {
#         "subject": "Apple Inc.",
#         "subject_gold": "APPLE_TECH",
#         "predicate": "sells",
#         "object": "Macbook",
#         "object_gold": "MACBOOK"
#     },
#     # =========================
#     # 2. Abbreviation Ambiguity
#     # =========================
#     {
#         "subject": "US",
#         "subject_gold": "UNITED_STATES",
#         "predicate": "invaded",
#         "object": "Iraq",
#         "object_gold": "IRAQ"
#     },
#     {
#         "subject": "US",
#         "subject_gold": "US_TENNIS_PLAYER",
#         "predicate": "won",
#         "object": "US Open",
#         "object_gold": "US_OPEN_TOURNAMENT"
#     },
#     {
#         "subject": "United States",
#         "subject_gold": "UNITED_STATES",
#         "predicate": "has capital",
#         "object": "Washington DC",
#         "object_gold": "WASHINGTON_DC"
#     },

#     # =========================
#     # 3. Semantic Embedding Trap
#     # =========================
#     {
#         "subject": "Amazon",
#         "subject_gold": "AMAZON_COMPANY",
#         "predicate": "develops",
#         "object": "Alexa",
#         "object_gold": "ALEXA"
#     },
#     {
#         "subject": "Amazon",
#         "subject_gold": "AMAZON_COMPANY",
#         "predicate": "headquartered in",
#         "object": "Seattle",
#         "object_gold": "SEATTLE"
#     },
#     {
#         "subject": "Amazon River",
#         "subject_gold": "AMAZON_RIVER",
#         "predicate": "flows through",
#         "object": "Brazil",
#         "object_gold": "BRAZIL"
#     },

#     # =========================
#     # 4. Role vs Entity Confusion
#     # =========================
#     {
#         "subject": "President",
#         "subject_gold": "US_PRESIDENT_ROLE",
#         "predicate": "leads",
#         "object": "United States",
#         "object_gold": "UNITED_STATES"
#     },
#     {
#         "subject": "Joe Biden",
#         "subject_gold": "JOE_BIDEN",
#         "predicate": "is",
#         "object": "President",
#         "object_gold": "US_PRESIDENT_ROLE"
#     },
#     {
#         "subject": "Joe Biden",
#         "subject_gold": "JOE_BIDEN",
#         "predicate": "born in",
#         "object": "Scranton",
#         "object_gold": "SCRANTON_PA"
#     },

#     # =========================
#     # 5. Predicate Paraphrase
#     # =========================
#     {
#         "subject": "Elon Musk",
#         "subject_gold": "ELON_MUSK",
#         "predicate": "leads",
#         "object": "SpaceX",
#         "object_gold": "SPACEX"
#     },
#     {
#         "subject": "Elon Musk",
#         "subject_gold": "ELON_MUSK",
#         "predicate": "is CEO of",
#         "object": "SpaceX",
#         "object_gold": "SPACEX"
#     },
#     {
#         "subject": "Elon Musk",
#         "subject_gold": "ELON_MUSK",
#         "predicate": "runs",
#         "object": "SpaceX",
#         "object_gold": "SPACEX"
#     },

#     # =========================
#     # 6. Predicate Polarity Conflict
#     # =========================
#     {
#         "subject": "Company X",
#         "subject_gold": "COMPANY_X",
#         "predicate": "acquired",
#         "object": "Company Y",
#         "object_gold": "COMPANY_Y"
#     },
#     {
#         "subject": "Company Y",
#         "subject_gold": "COMPANY_Y",
#         "predicate": "acquired",
#         "object": "Company X",
#         "object_gold": "COMPANY_X"
#     },

#     # =========================
#     # 7. Structural / Neighborhood Deception
#     # =========================
#     {
#         "subject": "Paris",
#         "subject_gold": "PARIS_FRANCE",
#         "predicate": "located in",
#         "object": "France",
#         "object_gold": "FRANCE"
#     },
#     {
#         "subject": "Paris",
#         "subject_gold": "PARIS_FRANCE",
#         "predicate": "has river",
#         "object": "Seine",
#         "object_gold": "SEINE_RIVER"
#     },
#     {
#         "subject": "Paris",
#         "subject_gold": "PARIS_TEXAS",
#         "predicate": "located in",
#         "object": "Texas",
#         "object_gold": "TEXAS"
#     },
#     {
#         "subject": "Paris",
#         "subject_gold": "PARIS_TEXAS",
#         "predicate": "has population",
#         "object": "25000",
#         "object_gold": "POPULATION_25000"
#     },

#     # =========================
#     # 8. Copycat Organization
#     # =========================
#     {
#         "subject": "OpenAI",
#         "subject_gold": "OPENAI",
#         "predicate": "develops",
#         "object": "ChatGPT",
#         "object_gold": "CHATGPT"
#     },
#     {
#         "subject": "OpenAI Research",
#         "subject_gold": "OPENAI_RESEARCH",
#         "predicate": "develops",
#         "object": "ChatGPT",
#         "object_gold": "CHATGPT"
#     },
#     {
#         "subject": "OpenAI Research",
#         "subject_gold": "OPENAI_RESEARCH",
#         "predicate": "published",
#         "object": "Paper A",
#         "object_gold": "PAPER_A"
#     },
#     {
#         "subject": "OpenAI",
#         "subject_gold": "OPENAI",
#         "predicate": "published",
#         "object": "Paper B",
#         "object_gold": "PAPER_B"
#     },

#     # =========================
#     # 9. Temporal Drift
#     # =========================
#     {
#         "subject": "Google",
#         "subject_gold": "GOOGLE",
#         "predicate": "CEO",
#         "object": "Larry Page",
#         "object_gold": "LARRY_PAGE"
#     },
#     {
#         "subject": "Google",
#         "subject_gold": "GOOGLE",
#         "predicate": "CEO",
#         "object": "Sundar Pichai",
#         "object_gold": "SUNDAR_PICHAI"
#     },

#     # =========================
#     # 10. Compound Adversarial Case
#     # =========================
#     {
#         "subject": "Meta",
#         "subject_gold": "META",
#         "predicate": "formerly known as",
#         "object": "Facebook",
#         "object_gold": "FACEBOOK"
#     },
#     {
#         "subject": "Facebook",
#         "subject_gold": "FACEBOOK",
#         "predicate": "owns",
#         "object": "Instagram",
#         "object_gold": "INSTAGRAM"
#     },
#     {
#         "subject": "Meta AI",
#         "subject_gold": "META_AI",
#         "predicate": "develops",
#         "object": "LLaMA",
#         "object_gold": "LLAMA_MODEL"
#     },
#     {
#         "subject": "Facebook AI Research",
#         "subject_gold": "FAIR",
#         "predicate": "develops",
#         "object": "LLaMA",
#         "object_gold": "LLAMA_MODEL"
#     }
# ]

In [89]:
from transformers import pipeline

ner_pipeline = pipeline(
    "ner",
    model="dslim/bert-base-NER",
    aggregation_strategy="simple"
)

Some weights of the model checkpoint at dslim/bert-base-NER were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use mps:0


In [90]:
# unique_entities = sorted(unique_entities)

# emb = model.encode(unique_entities, convert_to_tensor=False)

# # Normalize embeddings for cosine similarity
# faiss.normalize_L2(emb)

# dim = emb.shape[1]

# # Inner product index (cosine similarity)
# faiss_index = faiss.IndexFlatIP(dim)

# base_index = faiss.IndexFlatIP(dim)
# faiss_index = faiss.IndexIDMap(base_index)

# faiss_index.add_with_ids(
#     emb,
#     np.array(unique_entities, dtype=np.int64)
# )


unique_entities = sorted(unique_entities)

str2int = {eid: i for i, eid in enumerate(unique_entities)}
int2str = {i: eid for eid, i in str2int.items()}

emb = model.encode(unique_entities, convert_to_tensor=False)
emb = emb.astype("float32")
faiss.normalize_L2(emb)

dim = emb.shape[1]

base_index = faiss.IndexFlatIP(dim)
faiss_index = faiss.IndexIDMap(base_index)

faiss_index.add_with_ids(
    emb,
    np.array([str2int[eid] for eid in unique_entities], dtype=np.int64)
)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [91]:
str2int

{'Apple': 0,
 'Apple Inc.': 1,
 'Fruit': 2,
 'MacBook': 3,
 'September': 4,
 'iPhone 15': 5}

In [92]:
int2str

{0: 'Apple',
 1: 'Apple Inc.',
 2: 'Fruit',
 3: 'MacBook',
 4: 'September',
 5: 'iPhone 15'}

In [93]:
# ----------------------------
# Initialize entity store
# ----------------------------

entity_store: dict = {}
entity_ids: list = []
entity_id_counter: int = 1

existing_edge_keys = set()  # to track duplicates
accepted_edges: list = []
excluded_edges: list = []

In [94]:
# ----------------------------
# Helper functions
# ----------------------------

from collections import defaultdict

def normalize(v):
    """
    L2 normalize a vector
    """
    return v / np.linalg.norm(v)

def check_entity_store(entity_name: str, entity_store: dict):
    
    reverse_dict = defaultdict(list)
    
    for k, v in entity_store.items():
        reverse_dict[v['name']].append(k)

    keys = reverse_dict[entity_name]

    return keys


# ----------------------------
# Core functions
# ----------------------------

def string_sim(a: str, b: str) -> float:
    """Normalized string similarity between 0 and 1"""
    return fuzz.token_sort_ratio(a, b) / 100

def neighbor_overlap(new_neighbors: set, existing_neighbors: set) -> float:

    """
    Calculate Jaccard similarity between new_neighbors and existing_neighbors sets.
        - new_neighbors: set of "predicate:entity_id" for the new entity
        - existing_neighbors: set of "predicate:entity_id" for the existing entity
        Returns a similarity score between 0 and 1.
    """

    print("new_neighbors:", new_neighbors)
    print("existing_neighbors:", existing_neighbors)

    if not new_neighbors or not existing_neighbors:
        return 0.0
    intersection = new_neighbors.intersection(existing_neighbors)
    union = new_neighbors.union(existing_neighbors)

    return len(intersection) / len(union)

def semantic_blocking(entity_text, entity_store, top_k=10, sim_threshold=0.7):
    """
    Semantic blocking using FAISS for efficient retrieval of candidate entities based on embedding similarity.
    """
    
    if faiss_index.ntotal == 0:
        return []

    # query_emb = model.encode(entity_text)
    # query_emb = normalize(query_emb).reshape(1, -1)
    # scores, indices = faiss_index.search(query_emb, top_k)

    query_emb = model.encode(entity_text, convert_to_tensor=False)
    # Ensure numpy array
    query_emb = np.asarray(query_emb, dtype="float32")
    # Make it 2D FIRST
    query_emb = query_emb.reshape(1, -1)
    # Normalize for cosine similarity
    faiss.normalize_L2(query_emb)

    # print(type(query_emb), query_emb.shape)
    # print(type(top_k), top_k)
    # scores, indices = faiss_index.search(query_emb, top_k)

    # candidates = []

    scores, indices = faiss_index.search(query_emb, top_k)

    candidates = []
    for score, int_id in zip(scores[0], indices[0]):
        if int_id == -1:
            continue
        if score >= sim_threshold:
            candidates.append({
                "entity_id": int2str[int(int_id)],
                "e_sim": float(score)
            })
            
    return candidates


def predict_entity_type(entity_text, predicate=None):
    """
    Predict coarse entity type using a pretrained NER model.
    Returns: PERSON | ORGANIZATION | LOCATION | MISC | UNKNOWN
    """
    # Add minimal context (important!)
    if predicate:
        text = f"{entity_text} {predicate}"
    else:
        text = entity_text

    try:
        entities = ner_pipeline(text)
    except Exception:
        return "UNKNOWN"

    for ent in entities:
        if ent["word"].lower() in entity_text.lower():
            label = ent["entity_group"]
            print(f"NER prediction for '{entity_text}': {label}")
            if label == "PER":
                return "PERSON"
            if label == "ORG":
                return "ORGANIZATION"
            if label == "LOC":
                return "LOCATION"
            if label == "MISC":
                return "MISC"

    return "UNKNOWN"

def types_compatible(type1, type2):
    """
    Simple type compatibility check.
    If either type is UNKNOWN, we consider them compatible (fallback). Otherwise, they must match exactly.
    """
    if type1 == "UNKNOWN" or type2 == "UNKNOWN":
        return True  # fallback
    return type1 == type2


def composite_similarity(entity_text, existing, predicate=None, other_entity_id=None,
                         alpha=0.4, beta=0.3, gamma=0.2, delta=0.1):
    """
    Composite similarity for entity resolution.
    Weights sum to 1: alpha + beta + gamma + delta = 1
    """
    # String similarity
    s_sim = string_sim(entity_text, existing["name"])

    # Embedding similarity (contextual)
    # Use the triple text as context for embeddings
    context_text = f"{entity_text} {predicate}" if predicate else entity_text
    emb_entity = model.encode([context_text])
    emb_existing = model.encode([existing["name"]])
    e_sim = util.cos_sim(emb_entity, emb_existing).item()  # cosine similarity

    # Neighbor similarity
    new_neighbors = set()
    if other_entity_id is not None and predicate is not None:
        new_neighbors.add(f"{predicate}:{other_entity_id}")
    n_sim = neighbor_overlap(new_neighbors, existing["neighbors"])

    # Predicate/type similarity
    type_match = types_compatible(predict_entity_type(entity_text, predicate), existing["type"])
    p_sim = 1.0 if type_match else 0.0

    print(f"Composite similarity components for '{entity_text}' vs '{existing['name']}': s_sim={s_sim}, e_sim={e_sim}, n_sim={n_sim}, p_sim={p_sim}")

    # Composite score
    composite_score = alpha * s_sim + beta * e_sim + gamma * n_sim + delta * p_sim
    return composite_score


def resolve_entity(entity_text, role, predicate, other_entity_id=None):

    """
    Resolve an entity by finding the best candidate to MERGE with or deciding to INSERT a new entity.
    """
    global entity_id_counter

    ## Select only a block of candidates to reduce comparison of all entities
    candidates = semantic_blocking(entity_text, entity_store)
    print(f"Resolving '{entity_text}' with predicate '{predicate}' and role '{role}'")
    print("Candidates:", candidates)
    best_candidate = None
    best_score = 0.0
    print(type(candidates), candidates)
    new_type = predict_entity_type(entity_text, predicate)

    ## Check each candidate
    for c in candidates:
        print(f"checking candidate {c} for entity {entity_text}")
        
        # I need to fix this part. The candidate c is a dict with keys "entity_id" and "e_sim". I need to retrieve the actual entity from the entity_store using c["entity_id"].
        # The entity_store is empty at the beginning, so I need to make sure that I populate it with the initial entities before I can use it for resolution. For now, I will just print out the candidate and the entity store to debug.

        ## Check if candidate exists already exists in the entity store.
        existing_id_list = check_entity_store(c['entity_id'], entity_store)

        if len(existing_id_list) > 0:
            existing_id = existing_id_list[0]
        else:
            continue

        # existing = entity_store[c["entity_id"]]
        existing = entity_store[existing_id]
        existing_type = existing["type"]

        # HARD TYPE GATE
        if not types_compatible(new_type, existing_type):
            continue

        score = composite_similarity(
            entity_text,
            existing,
            predicate=predicate,
            other_entity_id=other_entity_id
        )

        print(f"score: {score} and best score {best_score}")
        if score > best_score and score >= 0.65:
            best_candidate = existing_id
            # best_candidate = c["entity_id"]
            best_score = score

        if best_candidate is not None:
            return best_candidate, "MERGE"

    # INSERT
    new_id = f"E{entity_id_counter}"
    entity_id_counter += 1
    entity_store[new_id] = {
        "name": entity_text,
        "type": new_type,
        "neighbors": set()
    }
    return new_id, "INSERT"

def insert_edge(subject_id, predicate, object_id):
    """
    Insert edge between subject and object, and update their neighbors.
    This function is called after deciding to accept the edge (no duplicate).
    """
    entity_store[subject_id]["neighbors"].add(f"{predicate}:{object_id}")
    entity_store[object_id]["neighbors"].add(f"{predicate}:{subject_id}")


Pipeline

In [95]:
for triple in toy_data:
    subject_text = triple["subject"]
    object_text = triple["object"]
    predicate = triple["predicate"]

    print("========================")
    print("Resolving triple:", triple)
    # resolve subject first
    subj_id, subj_action = resolve_entity(
        subject_text,
        role="subject",
        predicate=predicate
    )

    # resolve object (can use subject as neighbor)
    obj_id, obj_action = resolve_entity(
        object_text,
        role="object",
        predicate=predicate,
        other_entity_id=subj_id
    )

    edge_key = (subj_id, predicate, obj_id)

    if edge_key in existing_edge_keys:
        excluded_edges.append({
            **triple,
            "reason": "duplicate_edge",
            "subject_id": subj_id,
            "object_id": obj_id
        })
        continue

    accepted_edges.append({
        **triple,
        "subject_id": subj_id,
        "object_id": obj_id,
        "subject_action": subj_action,
        "object_action": obj_action
    })

    existing_edge_keys.add(edge_key)

    print("entity store current values")
    for k,v in entity_store.items():
        print(k,v)

    print("subj_id", subj_id)

    # update neighbors (MERGE semantics)
    entity_store[subj_id]["neighbors"].add(f"{predicate}:{obj_id}")
    entity_store[obj_id]["neighbors"].add(f"{predicate}:{subj_id}")

# ----------------------------
# Print results
# ----------------------------
print("=== Accepted Edges ===")
for e in accepted_edges:
    print(e)
    
print("\n=== Excluded Edges ===")
for e in excluded_edges:
    print(e)

print("\n=== Entity Store ===")
for eid, info in entity_store.items():
    print(eid, info['name'], info['neighbors'], info['type'])

Resolving triple: {'subject': 'Apple', 'subject_gold': 'APPLE_TECH', 'predicate': 'released', 'object': 'iPhone 15', 'object_gold': 'IPHONE_15'}


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Resolving 'Apple' with predicate 'released' and role 'subject'
Candidates: [{'entity_id': 'Apple', 'e_sim': 1.0}, {'entity_id': 'Apple Inc.', 'e_sim': 0.7887802720069885}]
<class 'list'> [{'entity_id': 'Apple', 'e_sim': 1.0}, {'entity_id': 'Apple Inc.', 'e_sim': 0.7887802720069885}]
NER prediction for 'Apple': ORG
checking candidate {'entity_id': 'Apple', 'e_sim': 1.0} for entity Apple
checking candidate {'entity_id': 'Apple Inc.', 'e_sim': 0.7887802720069885} for entity Apple


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Resolving 'iPhone 15' with predicate 'released' and role 'object'
Candidates: [{'entity_id': 'iPhone 15', 'e_sim': 1.0000001192092896}]
<class 'list'> [{'entity_id': 'iPhone 15', 'e_sim': 1.0000001192092896}]
NER prediction for 'iPhone 15': MISC
checking candidate {'entity_id': 'iPhone 15', 'e_sim': 1.0000001192092896} for entity iPhone 15
entity store current values
E1 {'name': 'Apple', 'type': 'ORGANIZATION', 'neighbors': set()}
E2 {'name': 'iPhone 15', 'type': 'MISC', 'neighbors': set()}
subj_id E1
Resolving triple: {'subject': 'Apple', 'subject_gold': 'APPLE_FRUIT', 'predicate': 'harvested in', 'object': 'September', 'object_gold': 'MONTH_SEPTEMBER'}


...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

new_neighbors: set()
existing_neighbors: {'is a type of:E5', 'released:E2'}
NER prediction for 'Apple Inc.': ORG
Composite similarity components for 'Apple Inc.' vs 'Apple': s_sim=0.71, e_sim=0.6501150131225586, n_sim=0.0, p_sim=1.0
score: 0.5790345039367676 and best score 0.0


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Resolving 'MacBook' with predicate 'sells' and role 'object'
Candidates: [{'entity_id': 'MacBook', 'e_sim': 0.9999998807907104}]
<class 'list'> [{'entity_id': 'MacBook', 'e_sim': 0.9999998807907104}]
NER prediction for 'MacBook': ORG
checking candidate {'entity_id': 'MacBook', 'e_sim': 0.9999998807907104} for entity MacBook
entity store current values
E1 {'name': 'Apple', 'type': 'ORGANIZATION', 'neighbors': {'is a type of:E5', 'released:E2'}}
E2 {'name': 'iPhone 15', 'type': 'MISC', 'neighbors': {'released:E1'}}
E3 {'name': 'Apple', 'type': 'UNKNOWN', 'neighbors': {'harvested in:E4'}}
E4 {'name': 'September', 'type': 'UNKNOWN', 'neighbors': {'harvested in:E3'}}
E5 {'name': 'Fruit', 'type': 'UNKNOWN', 'neighbors': {'is a type of:E1'}}
E6 {'name': 'Apple Inc.', 'type': 'ORGANIZATION', 'neighbors': set()}
E7 {'name': 'MacBook', 'type': 'ORGANIZATION', 'neighbors': set()}
subj_id E6
=== Accepted Edges ===
{'subject': 'Apple', 'subject_gold': 'APPLE_TECH', 'predicate': 'released', 'object'

## Evaluation

### Evaluation functions

(Note): There is something wrong with clustering. It also associates the objects with the cluster of the subjects.

Fix:
- Add a separate gold cluster for objects.

In [96]:
from collections import defaultdict
from itertools import combinations

# ----------------------------
# Helper: Build entity → gold clusters mapping
# ----------------------------

def build_gold_entity_map(gold_clusters):
    """
    Maps entity_text -> set of gold clusters it belongs to
    """
    gold_entity_map = defaultdict(set)
    for gold_cluster, entities in gold_clusters.items():
        for e in entities:
            gold_entity_map[e].add(gold_cluster)
    return gold_entity_map

# ----------------------------
# Helper: Generate all pairwise links for evaluation
# ----------------------------
def pairwise_links(items):
    """
    Returns all pairwise combinations (unordered) of items.
    Singleton clusters are treated as a self-link so they contribute to metrics.
    """
    if len(items) == 0:
        return set()
    if len(items) == 1:
        return {frozenset(items)}  # singleton contributes one link
    return set(frozenset([a, b]) for a, b in combinations(items, 2))

# ----------------------------
# Evaluation function
# ----------------------------
def evaluate_cluster_alignment(gold_clusters, predicted_clusters):
    gold_entity_map = build_gold_entity_map(gold_clusters)

    results = []

    for pred_cluster_id, pred_entities in predicted_clusters.items():
        matched_gold_clusters = defaultdict(set)

        for entity in pred_entities:
            golds = gold_entity_map.get(entity, set())
            if not golds:
                matched_gold_clusters["UNSEEN"].add(entity)
            else:
                for g in golds:
                    matched_gold_clusters[g].add(entity)

        results.append({
            "predicted_cluster": pred_cluster_id,
            "predicted_entities": pred_entities,
            "matched_gold_clusters": dict(matched_gold_clusters),
        })

    for r in results:
        print("=" * 50)
        print(f"Predicted Cluster: {r['predicted_cluster']}")
        print(f"Entities: {r['predicted_entities']}")

        golds = r["matched_gold_clusters"]

        if len(golds) == 1:
            gold_cluster = next(iter(golds))
            print(f"Pure match with gold cluster: {gold_cluster}")
        elif len(golds) > 1:
            print("Mixed cluster! Matches multiple gold clusters:")
            for g, ents in golds.items():
                print(f"  - {g}: {ents}")
        else:
            print("No gold cluster matched")
   
def evaluate_er_pipeline(resolved_entities, toy_dataset):
    """
    Evaluate entity resolution performance.
    
    resolved_entities: dict mapping entity_text -> resolved_entity_id
    toy_dataset: list of dicts with subject_gold / object_gold
    """

    # -------------------------
    # Build gold clusters (entity-level)
    # -------------------------
    gold_clusters = defaultdict(set)
    for triple in toy_dataset:
        gold_clusters[triple["subject_gold"]].add(triple["subject"])
        gold_clusters[triple["object_gold"]].add(triple["object"])

    print("Gold Clusters:")
    for cluster_id, entities in gold_clusters.items():
        print(f"{cluster_id}: {entities}")

    # -------------------------
    # Build predicted clusters
    # -------------------------
    pred_clusters = defaultdict(set)
    for entity_text, eid in resolved_entities.items():
        entity_text = entity_text.split("_")[0]  # remove type suffix for comparison
        pred_clusters[eid].add(entity_text)

    print("\nPredicted Clusters:")
    for cluster_id, entities in pred_clusters.items():
        print(f"{cluster_id}: {entities}")

    # -------------------------
    # Build pairwise links
    # -------------------------
    gold_links = set()
    for entities in gold_clusters.values():
        gold_links.update(pairwise_links(entities))

    pred_links = set()
    for entities in pred_clusters.values():
        pred_links.update(pairwise_links(entities))

    # -------------------------
    # Compute metrics
    # -------------------------
    true_positive = len(gold_links & pred_links)
    false_positive = len(pred_links - gold_links)
    false_negative = len(gold_links - pred_links)

    precision = true_positive / (true_positive + false_positive + 1e-10)
    recall = true_positive / (true_positive + false_negative + 1e-10)
    f1 = 2 * precision * recall / (precision + recall + 1e-10)

    print("\nEvaluation Results:")
    print(f"Pairwise Precision: {precision:.3f}")
    print(f"Pairwise Recall   : {recall:.3f}")
    print(f"Pairwise F1       : {f1:.3f}")

    # -------------------------
    # Problematic merges (over-merging)
    # -------------------------

    print(evaluate_cluster_alignment(gold_clusters, pred_clusters))

### Steps in using evaluation
Step 1. Create resolved entities variable for evaluation

In [97]:
for i in entity_store:
    print(i, entity_store[i])

E1 {'name': 'Apple', 'type': 'ORGANIZATION', 'neighbors': {'is a type of:E5', 'released:E2'}}
E2 {'name': 'iPhone 15', 'type': 'MISC', 'neighbors': {'released:E1'}}
E3 {'name': 'Apple', 'type': 'UNKNOWN', 'neighbors': {'harvested in:E4'}}
E4 {'name': 'September', 'type': 'UNKNOWN', 'neighbors': {'harvested in:E3'}}
E5 {'name': 'Fruit', 'type': 'UNKNOWN', 'neighbors': {'is a type of:E1'}}
E6 {'name': 'Apple Inc.', 'type': 'ORGANIZATION', 'neighbors': {'sells:E7'}}
E7 {'name': 'MacBook', 'type': 'ORGANIZATION', 'neighbors': {'sells:E6'}}


In [98]:
resolved_entities = {}

for eid, info in entity_store.items():
    entity_name = info["name"] + "_" + info["type"]  # include type for disambiguation
    resolved_entities[entity_name] = eid
    
print(resolved_entities)

{'Apple_ORGANIZATION': 'E1', 'iPhone 15_MISC': 'E2', 'Apple_UNKNOWN': 'E3', 'September_UNKNOWN': 'E4', 'Fruit_UNKNOWN': 'E5', 'Apple Inc._ORGANIZATION': 'E6', 'MacBook_ORGANIZATION': 'E7'}


Step 2: Call function

In [99]:
evaluate_er_pipeline(resolved_entities, toy_data)

Gold Clusters:
APPLE_TECH: {'Apple', 'Apple Inc.'}
IPHONE_15: {'iPhone 15'}
APPLE_FRUIT: {'Apple'}
MONTH_SEPTEMBER: {'September'}
FRUIT: {'Fruit'}
MACBOOK: {'MacBook'}

Predicted Clusters:
E1: {'Apple'}
E2: {'iPhone 15'}
E3: {'Apple'}
E4: {'September'}
E5: {'Fruit'}
E6: {'Apple Inc.'}
E7: {'MacBook'}

Evaluation Results:
Pairwise Precision: 0.833
Pairwise Recall   : 0.833
Pairwise F1       : 0.833
Predicted Cluster: E1
Entities: {'Apple'}
Mixed cluster! Matches multiple gold clusters:
  - APPLE_FRUIT: {'Apple'}
  - APPLE_TECH: {'Apple'}
Predicted Cluster: E2
Entities: {'iPhone 15'}
Pure match with gold cluster: IPHONE_15
Predicted Cluster: E3
Entities: {'Apple'}
Mixed cluster! Matches multiple gold clusters:
  - APPLE_FRUIT: {'Apple'}
  - APPLE_TECH: {'Apple'}
Predicted Cluster: E4
Entities: {'September'}
Pure match with gold cluster: MONTH_SEPTEMBER
Predicted Cluster: E5
Entities: {'Fruit'}
Pure match with gold cluster: FRUIT
Predicted Cluster: E6
Entities: {'Apple Inc.'}
Pure match w